In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
from scipy.stats import binom
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly
import kaleido
import plotly.io as pio

In [ ]:
#pip install scipy
#pip install plotly
#pip install nbformat
#!pip install --upgrade "plotly[kaleido]"
#!{sys.executable} -m pip install --upgrade kaleido


In [2]:
def bisection_prediction_set(
    X,
    Y,
    lower_bound_fn,
    upper_bound_fn,
    i_min,
    i_max,
    eps,
    alpha,
    i_start=None,
    tol=1e-4,
    max_iter=100,
    verbose=False,
):
    """
    Perform bisection over the prediction-set scale.

    At each tested value, test

        H0: miscoverage >= eps
        H1: miscoverage < eps

    If i_start is provided, the first test is performed
    at i_start. After that, ordinary bisection is used.

    Bisection direction:

        rejected
            -> current value works
            -> search SMALLER values

        not rejected
            -> current value does not work
            -> search LARGER values
    """

    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)

    c = len(Y)

    low_i = float(i_min)
    high_i = float(i_max)

    # --------------------------------------------------
    # Check starting value
    # --------------------------------------------------

    if i_start is not None:

        i_start = float(i_start)

        if not low_i <= i_start <= high_i:
            raise ValueError(
                f"i_start={i_start} must lie inside "
                f"[{low_i}, {high_i}]."
            )

    losses = {}
    p_values = {}
    rejected_i = []

    best_rejection = None

    for iteration in range(max_iter):

        # --------------------------------------------------
        # Stop when interval is sufficiently small
        # --------------------------------------------------

        if high_i - low_i <= tol:
            break

        # --------------------------------------------------
        # Choose value to test
        #
        # First iteration:
        #     use i_start if provided
        #
        # Later iterations:
        #     use ordinary midpoint
        # --------------------------------------------------

        if iteration == 0 and i_start is not None:
            i_mid = i_start

        else:
            i_mid = (low_i + high_i) / 2.0

        # --------------------------------------------------
        # Construct prediction set
        # --------------------------------------------------

        lower_i = np.asarray(
            lower_bound_fn(X, i_mid)
        ).reshape(-1)

        upper_i = np.asarray(
            upper_bound_fn(X, i_mid)
        ).reshape(-1)

        # --------------------------------------------------
        # Empirical miscoverage
        # --------------------------------------------------

        empirical_loss = np.mean(
            (Y < lower_i) | (Y > upper_i)
        )

        # --------------------------------------------------
        # Hypothesis-test p-value
        # --------------------------------------------------

        p_value = compute_p_value_binomial(
            empirical_loss=empirical_loss,
            c=c,
            epsilon=eps,
            alpha=alpha,
            a=i_mid,
        )

        rejected = p_value <= alpha

        # --------------------------------------------------
        # Store tested value
        # --------------------------------------------------

        losses[i_mid] = empirical_loss
        p_values[i_mid] = p_value

        if rejected:
            rejected_i.append(i_mid)

        if verbose:
            print(
                f"Iteration {iteration + 1}: "
                f"i = {i_mid:.6f}, "
                f"loss = {empirical_loss:.6f}, "
                f"p = {p_value:.6g}, "
                f"rejected = {rejected}"
            )

        # --------------------------------------------------
        # Bisection update
        # --------------------------------------------------

        if rejected:

            # Current value works.
            # Save it and try a SMALLER value.

            best_rejection = i_mid
            high_i = i_mid

        else:

            # Current value does not work.
            # Need a LARGER value.

            low_i = i_mid

    return (
        losses,
        p_values,
        rejected_i,
        best_rejection,
    )

In [3]:
##required to make the generated datasets


# this one uses the specified architecture to make the features
def real_data_model(data_dimension, noise_dim, architecture):

    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(noise_dim,)))

    for units, activation in architecture:
        model.add(layers.Dense(units))
        model.add(layers.Activation(activation))

    model.add(layers.Dense(data_dimension))
    model.add(layers.Lambda(lambda x: 10*tf.math.tanh(4*x)))

    model(tf.zeros((1, noise_dim)))
    return model

# this sets the weights and fixes them.
def set_random_scaled_weights(model, seed=None):
    if seed is not None:
        np.random.seed(seed)

    dense_layers = [
        layer for layer in model.layers
        if isinstance(layer, layers.Dense)
    ]

    for dense in dense_layers:
        W_shape = dense.get_weights()[0].shape
        b_shape = dense.get_weights()[1].shape

        W = np.random.normal(
            loc=0.0,
            scale=1 / np.sqrt(W_shape[0]),
            size=W_shape
        ).astype(np.float32)

        b = np.zeros(b_shape, dtype=np.float32)

        dense.set_weights([W, b])
        

        
#
def generate_feature_datasets(
    real_data_architectures,
    data_dimension,
    noise_dim,
    n,
    c,
    seed_base=2026,
    verbose=True,
):
    real_generators = []
    generated_datasets = []

    X_train_all = []
    X_cal_all = []

    for data_id, arch in enumerate(real_data_architectures):

        seed = seed_base + data_id

        if verbose:
            print("\n==============================")
            print(f"Dataset {data_id}")
            print(f"Seed = {seed}")
            print("==============================")

        tf.keras.utils.set_random_seed(seed)

        real_gen = real_data_model(
            data_dimension=data_dimension,
            noise_dim=noise_dim,
            architecture=arch,
        )

        set_random_scaled_weights(real_gen, seed=seed)
        real_generators.append(real_gen)

        noise = tf.random.stateless_normal(
            (n + c, noise_dim),
            seed=[seed, 0],
        )

        data = real_gen(noise, training=False).numpy()
        generated_datasets.append(data)

        X_train_all.append(data[:n])
        X_cal_all.append(data[n:n+c])

        if verbose:
            print(f"Feature data shape = {data.shape}")

    return (
        real_generators,
        generated_datasets,
        X_train_all,
        X_cal_all,
    )


#this uses the input dimensions, and  a random variable as input and gives the desired dimensional output for the response
def build_y_generator(
    input_dim=3,
    output_dim=1,
    architecture=[(32, "relu"), (32, "relu")],
    final_activation=None,
):
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for units, activation in architecture:
        model.add(layers.Dense(units))
        model.add(layers.Activation(activation))

    model.add(layers.Dense(output_dim, activation=final_activation))

    model(tf.zeros((1, input_dim)))
    return model

def generate_architecture_datasets(
    real_data_architectures,
    data_dimension,
    noise_dim,
    n,
    c,
    y1_generator_architecture,
    y2_generator_architecture,
    seed_base=2026,
    verbose=True,
):
    # --------------------------------------------------
    # Generate feature datasets
    # --------------------------------------------------

    (
        real_generators,
        generated_datasets,
        X_train_all,
        X_cal_all,
    ) = generate_feature_datasets(
        real_data_architectures=real_data_architectures,
        data_dimension=data_dimension,
        noise_dim=noise_dim,
        n=n,
        c=c,
        seed_base=seed_base,
        verbose=verbose,
    )

    # --------------------------------------------------
    # Build response generators
    # --------------------------------------------------

    y1_generator = build_y_generator(
        input_dim=3,
        output_dim=1,
        architecture=y1_generator_architecture,
        final_activation=None,
    )

    y2_generator = build_y_generator(
        input_dim=3,
        output_dim=1,
        architecture=y2_generator_architecture,
        final_activation=None,
    )

    # Stores:
    #   [X, Y1, Y2]
    full_generated_datasets = []

    full_X_train_all = []
    full_X_cal_all = []

    full_y_train_all = []
    full_y_cal_all = []

    # --------------------------------------------------
    # Generate responses
    # --------------------------------------------------

    for data_id, data in enumerate(generated_datasets):

        seed = seed_base + data_id
        rng = np.random.default_rng(seed)

        x1 = data[:, 0]
        x2 = data[:, 1]

        z = rng.normal(size=len(data))
        y_input = np.column_stack([x1, x2, z])

        Y1 = y1_generator(y_input, training=False).numpy().reshape(-1)
        Y2 = y2_generator(y_input, training=False).numpy().reshape(-1)

        Y = np.column_stack([Y1, Y2])

        dataset_with_y = np.column_stack([data, Y])

        full_generated_datasets.append(dataset_with_y)

        full_X_train_all.append(data[:n])
        full_X_cal_all.append(data[n:n+c])

        full_y_train_all.append(Y[:n])
        full_y_cal_all.append(Y[n:n+c])

        if verbose:
            print(f"\nFeature Dataset {data_id}")

            print(f"Full dataset {data_id}: [X, Y1, Y2]")
            print(f"Shape = {dataset_with_y.shape}")
            print(dataset_with_y[:5])

    return (
        real_generators,
        generated_datasets,
        full_generated_datasets,
        full_X_train_all,
        full_X_cal_all,
        y1_generator,
        y2_generator,
        full_y_train_all,
        full_y_cal_all,
    )

In [4]:
# making the predictor function, and the training for it.

# predictor architecture
def predictor_model(input_dimension, output_dimension, architecture):
    """
    Predictor network f_hat : X -> Y.

    Parameters
    ----------
    input_dimension : int
        Dimension of X.

    output_dimension : int
        Dimension of Y (1 for scalar regression).

    architecture : list of tuples
        Example:
        [
            (128, "relu"),
            (256, "relu"),
            (128, "relu")
        ]
    """

    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(input_dimension,)))

    for units, activation in architecture:
        model.add(layers.Dense(units))
        model.add(layers.Activation(activation))

    model.add(layers.Dense(output_dimension))

    model(tf.zeros((1, input_dimension)))

    return model

def train_predictor_auto_loss(
    X_train,
    y_train,
    architecture,
    epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    validation_split=0.2,
    verbose=0,
):
    input_dimension = X_train.shape[1]

    if y_train.ndim == 1:
        output_dimension = 1
        loss_fn = "mse"
    else:
        output_dimension = y_train.shape[1]

        if output_dimension == 1:
            loss_fn = "mse"
        else:
            loss_fn = euclidean_loss

    model = predictor_model(
        input_dimension=input_dimension,
        output_dimension=output_dimension,
        architecture=architecture,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=loss_fn,
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        verbose=verbose,
    )

    return model, history

def get_architecture_for_dim(predictor_architectures, j):
    if isinstance(predictor_architectures, list):
        return predictor_architectures[j]
    return predictor_architectures
# main training loop function
def train_all_predictors(
    X_train_all,
    y_train_all,
    predictor_architectures=None,
    epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    validation_split=0.2,
    verbose=0,
):
    predictor_dim_models = []
    history_dim = []

    for data_id in range(len(X_train_all)):

        print("\n==============================")
        print(f"Training predictors: Dataset {data_id}")
        print("==============================")

        X_train = X_train_all[data_id]
        Y_train = y_train_all[data_id]

        if Y_train.ndim == 1:
            Y_train = Y_train.reshape(-1, 1)

        y_dim = Y_train.shape[1]

        if isinstance(predictor_architectures, list):
            if len(predictor_architectures) != y_dim:
                raise ValueError(
                    f"Expected {y_dim} predictor architectures, "
                    f"but got {len(predictor_architectures)}."
                )

        dataset_dim_models = []
        dataset_dim_histories = []

        # --------------------------------------------------
        # Train separate predictors: X -> Y1, X -> Y2, ...
        # --------------------------------------------------

        for j in range(y_dim):

            print(f"Training X -> Y{j + 1}")

            architecture_j = get_architecture_for_dim(
                predictor_architectures,
                j,
            )

            model_j, hist_j = train_predictor_auto_loss(
                X_train=X_train,
                y_train=Y_train[:, j],
                architecture=architecture_j,
                epochs=epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                validation_split=validation_split,
                verbose=verbose,
            )

            dataset_dim_models.append(model_j)
            dataset_dim_histories.append(hist_j)

        predictor_dim_models.append(dataset_dim_models)
        history_dim.append(dataset_dim_histories)

    return (
        predictor_dim_models,
        history_dim,
    )

In [5]:
def pinball_loss(tau):
    def loss(y_true, y_pred):
        e = y_true - y_pred
        return tf.reduce_mean(tf.maximum(tau * e, (tau - 1.0) * e))
    return loss

def add_activation_layer(x, activation):
    if activation == "leaky_relu":
        return tf.keras.layers.LeakyReLU()(x)
    else:
        return tf.keras.layers.Activation(activation)(x)


def build_positive_offset_network(input_dim, architecture):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = inputs

    for units, activation in architecture:
        x = tf.keras.layers.Dense(units)(x)
        x = add_activation_layer(x, activation)

    # Positive output
    outputs = tf.keras.layers.Dense(1, activation="softplus")(x)

    return tf.keras.Model(inputs, outputs)


def build_upper_quantile_model(predictor, input_dim, architecture):
    predictor.trainable = False

    inputs = tf.keras.Input(shape=(input_dim,))
    base = predictor(inputs)
    offset = build_positive_offset_network(input_dim, architecture)(inputs)

    outputs = base + offset

    return tf.keras.Model(inputs, outputs)


def build_lower_quantile_model(predictor, input_dim, architecture):
    predictor.trainable = False

    inputs = tf.keras.Input(shape=(input_dim,))
    base = predictor(inputs)
    offset = build_positive_offset_network(input_dim, architecture)(inputs)

    outputs = base - offset

    return tf.keras.Model(inputs, outputs)

In [6]:
def indicator_losses_over_sets(
    predictor,
    X,
    y,
    set_constructors,
):
    """
    Computes empirical indicator losses of the form

        1{ y_i not in S_i(X_i) }

    where each set constructor may depend on X, f_hat(X), and y.

    Parameters
    ----------
    predictor : fitted model
        Model producing f_hat(X).

    X : array-like
        Calibration features.

    y : array-like
        Calibration responses. Can be 1D or multidimensional.

    set_constructors : dict
        Dictionary mapping keys to functions.

        Each function should have signature:

            set_fn(X, y, y_pred)

        and return a boolean array of shape (n,), where True means
        y_i belongs to the constructed set.

    Returns
    -------
    losses : dict
        losses[key] = mean membership failure probability.
    """

    X = np.asarray(X)
    y = np.asarray(y)

    y_pred = predictor.predict(X, verbose=0)

    losses = {}

    for key, set_fn in set_constructors.items():

        in_set = np.asarray(set_fn(X, y, y_pred)).reshape(-1)

        if in_set.dtype != bool:
            raise ValueError(
                f"set_constructors[{key}] must return a boolean array."
            )

        if in_set.shape[0] != X.shape[0]:
            raise ValueError(
                f"set_constructors[{key}] returned length {in_set.shape[0]}, "
                f"but X has length {X.shape[0]}."
            )

        losses[key] = np.mean(~in_set)

    return losses


def compute_p_value_binomial(
    empirical_loss,
    c,
    epsilon,
    alpha=None,
    a=None,
):
   #binomial p value

    k = int(round(empirical_loss * c))

    p_value = binom.cdf(k, c, epsilon)

    return p_value

In [7]:
def compute_radius_grid(
    predictor,
    X_train,
    y_train,
    N,
    lower_scale=0.8,
    upper_scale=1.2,
    verbose=True,
):
    """
    Compute a radius grid from absolute training residuals
    for a one-dimensional response.

    Parameters
    ----------
    predictor : keras model
        Predictor for one response dimension.

    X_train : ndarray
        Training features.

    y_train : ndarray
        One-dimensional response values.

    N : int
        Number of grid intervals.

    lower_scale : float, default=0.8
        Multiplier applied to the minimum residual.

    upper_scale : float, default=1.2
        Multiplier applied to the maximum residual.

    verbose : bool, default=True
        Whether to print grid information.

    Returns
    -------
    a_min : float
        Minimum absolute residual.

    a_max : float
        Maximum absolute residual.

    a_values : ndarray
        Candidate radius values.
    """
    y_train = np.asarray(y_train).reshape(-1)

    y_train_pred = np.asarray(
        predictor.predict(
            X_train,
            verbose=0,
        )
    ).reshape(-1)

    if len(y_train_pred) != len(y_train):
        raise ValueError(
            "Predictions and response values must have the same length."
        )

    residuals = np.abs(
        y_train_pred - y_train
    )

    a_min = float(np.min(residuals))
    a_max = float(np.max(residuals))

    a_lower = lower_scale * a_min
    a_upper = upper_scale * a_max

    # Using linspace gives exactly N + 1 grid points.
    a_values = np.linspace(
        a_lower,
        a_upper,
        N + 1,
    )

    if verbose:
        print(
            f"a_min={a_min:.6f}, "
            f"a_max={a_max:.6f}, "
            f"num_radii={len(a_values)}"
        )

    return (
        a_min,
        a_max,
        a_values,
    )


def compute_radius_grids(
    predictor_models,
    X_train_all,
    y_train_all,
    N,
    lower_scale=0.8,
    upper_scale=1.2,
    verbose=True,
):
    """
    Compute a radius grid for each dataset for one
    response dimension.
    """
    num_datasets = len(X_train_all)

    if len(predictor_models) != num_datasets:
        raise ValueError(
            "predictor_models and X_train_all must have the same length."
        )

    if len(y_train_all) != num_datasets:
        raise ValueError(
            "y_train_all and X_train_all must have the same length."
        )

    a_min_all = []
    a_max_all = []
    a_values_all = []

    for data_id in range(num_datasets):

        if verbose:
            print(
                f"\nComputing radius grid: Dataset {data_id}"
            )

        (
            a_min,
            a_max,
            a_values,
        ) = compute_radius_grid(
            predictor=predictor_models[data_id],
            X_train=X_train_all[data_id],
            y_train=y_train_all[data_id],
            N=N,
            lower_scale=lower_scale,
            upper_scale=upper_scale,
            verbose=verbose,
        )

        a_min_all.append(a_min)
        a_max_all.append(a_max)
        a_values_all.append(a_values)

    return (
        a_min_all,
        a_max_all,
        a_values_all,
    )


def compute_all_radius_grids(
    predictor_dim_models,
    X_train_all,
    y_train_all,
    N,
    lower_scale=0.8,
    upper_scale=1.2,
    verbose=True,
):
    """
    Compute radius grids separately for each response
    dimension Y_j and each dataset.

    Returns
    -------
    a_min_dim_all : list
        a_min_dim_all[j][data_id] is the minimum residual
        for response dimension j in dataset data_id.

    a_max_dim_all : list
        a_max_dim_all[j][data_id] is the maximum residual
        for response dimension j in dataset data_id.

    a_values_dim_all : list
        a_values_dim_all[j][data_id] contains the radius
        grid for response dimension j in dataset data_id.
    """
    num_datasets = len(X_train_all)

    if len(y_train_all) != num_datasets:
        raise ValueError(
            "X_train_all and y_train_all must have the same length."
        )

    if len(predictor_dim_models) != num_datasets:
        raise ValueError(
            "predictor_dim_models and X_train_all must have "
            "the same length."
        )

    Y0 = np.asarray(y_train_all[0])

    if Y0.ndim == 1:
        y_dim = 1
    else:
        y_dim = Y0.shape[1]

    a_min_dim_all = []
    a_max_dim_all = []
    a_values_dim_all = []

    for j in range(y_dim):

        predictor_j_models = [
            predictor_dim_models[data_id][j]
            for data_id in range(num_datasets)
        ]

        yj_train_all = [
            (
                np.asarray(y_train_all[data_id]).reshape(-1)
                if np.asarray(y_train_all[data_id]).ndim == 1
                else np.asarray(y_train_all[data_id])[:, j]
            )
            for data_id in range(num_datasets)
        ]

        if verbose:
            print("\n==============================")
            print(f"Computing radius grids for Y{j + 1}")
            print("==============================")

        (
            a_min_j_all,
            a_max_j_all,
            a_values_j_all,
        ) = compute_radius_grids(
            predictor_models=predictor_j_models,
            X_train_all=X_train_all,
            y_train_all=yj_train_all,
            N=N,
            lower_scale=lower_scale,
            upper_scale=upper_scale,
            verbose=verbose,
        )

        a_min_dim_all.append(a_min_j_all)
        a_max_dim_all.append(a_max_j_all)
        a_values_dim_all.append(a_values_j_all)

    return (
        a_min_dim_all,
        a_max_dim_all,
        a_values_dim_all,
    )

In [8]:
def run_radius_hypothesis_test(
    predictor,
    X_cal,
    y_cal,
    a_values,
    eps,
    alpha,
    tol=1e-4,
    max_iter=100,
    verbose=False,
):
    """
    Run radius prediction-set hypothesis testing using bisection.

    Prediction set:

        C_a(x) = [f_hat(x) - a, f_hat(x) + a]

    The initial bisection interval is determined from
    min(a_values) and max(a_values).
    """

    X_cal = np.asarray(X_cal)
    y_cal = np.asarray(y_cal).reshape(-1)

    a_values = np.asarray(a_values, dtype=float)

    if a_values.size == 0:
        raise ValueError("a_values must contain at least one value.")

    # --------------------------------------------------
    # Compute predictor output ONCE
    # --------------------------------------------------

    f_hat = np.asarray(
        predictor.predict(
            X_cal,
            verbose=0,
        )
    ).reshape(-1)

    # --------------------------------------------------
    # Radius lower and upper bound functions
    # --------------------------------------------------

    def lower_bound_fn(X, a):
        return f_hat - a

    def upper_bound_fn(X, a):
        return f_hat + a

    # --------------------------------------------------
    # Initial bisection interval
    # --------------------------------------------------

    a_min = float(np.min(a_values))
    a_max = float(np.max(a_values))

    # --------------------------------------------------
    # Run bisection
    # --------------------------------------------------

    (
        losses,
        p_values,
        rejected_a,
        best_rejection,
    ) = bisection_prediction_set(
        X=X_cal,
        Y=y_cal,
        lower_bound_fn=lower_bound_fn,
        upper_bound_fn=upper_bound_fn,
        i_min=a_min,
        i_max=a_max,
        eps=eps,       # <- correct
        alpha=alpha,
        tol=tol,
        max_iter=max_iter,
        verbose=verbose,
    )

    return (
        losses,
        p_values,
        rejected_a,
        best_rejection,
    )

In [9]:
# Hypothesis tests for the models.
def run_radius_tests_for_models(
    predictor_models,
    X_cal_all,
    y_cal_all,
    a_values_all,
    eps,
    alpha,
    tol=1e-4,
    max_iter=100,
    verbose=True,
    label="",
):
    loss_results_all = []
    p_values_all = []
    rejections_all = []
    best_rejections = []

    for data_id in range(len(predictor_models)):

        if verbose:
            print("\n==============================")
            print(
                f"Calibration testing {label}: "
                f"Dataset {data_id}"
            )
            print("==============================")

        (
            losses,
            p_values,
            rejected_a,
            best_rejection,
        ) = run_radius_hypothesis_test(
            predictor=predictor_models[data_id],
            X_cal=X_cal_all[data_id],
            y_cal=y_cal_all[data_id],
            a_values=a_values_all[data_id],
            eps=eps,
            alpha=alpha,
            tol=tol,
            max_iter=max_iter,
            verbose=verbose,
        )

        loss_results_all.append(
            losses
        )

        p_values_all.append(
            p_values
        )

        rejections_all.append(
            rejected_a
        )

        best_rejections.append(
            best_rejection
        )

    return (
        loss_results_all,
        p_values_all,
        rejections_all,
        best_rejections,
    )
def run_all_radius_tests(
    predictor_dim_models,
    X_cal_all,
    y_cal_all,
    a_values_dim_all,
    eps,
    alpha,
    tol=1e-4,
    max_iter=100,
    verbose=True,
):
    """
    Run radius hypothesis tests separately for each
    response dimension Y_j and for each dataset,
    using bisection over the radius.
    """

    num_datasets = len(X_cal_all)

    if len(y_cal_all) != num_datasets:
        raise ValueError(
            "X_cal_all and y_cal_all must have the same length."
        )

    if len(predictor_dim_models) != num_datasets:
        raise ValueError(
            "predictor_dim_models and X_cal_all must have "
            "the same length."
        )

    if num_datasets == 0:
        raise ValueError(
            "At least one calibration dataset must be provided."
        )

    Y0 = np.asarray(y_cal_all[0])

    if Y0.ndim == 1:
        y_dim = 1

    elif Y0.ndim == 2:
        y_dim = Y0.shape[1]

    else:
        raise ValueError(
            "Each response array must be one-dimensional or "
            "two-dimensional."
        )

    if len(a_values_dim_all) != y_dim:
        raise ValueError(
            f"Expected radius grids for {y_dim} response dimensions, "
            f"but got {len(a_values_dim_all)}."
        )

    loss_results_dim_all = []
    p_values_dim_all = []
    rejections_dim_all = []
    best_rejections_dim = []

    # ==================================================
    # Loop over response dimensions
    # ==================================================

    for j in range(y_dim):

        predictor_j_models = [
            predictor_dim_models[data_id][j]
            for data_id in range(num_datasets)
        ]

        yj_cal_all = []

        # ----------------------------------------------
        # Extract Y_j from each dataset
        # ----------------------------------------------

        for data_id in range(num_datasets):

            y_data = np.asarray(
                y_cal_all[data_id]
            )

            if y_data.ndim == 1:

                if y_dim != 1:
                    raise ValueError(
                        f"Dataset {data_id} has one-dimensional "
                        f"responses, but expected {y_dim} dimensions."
                    )

                yj_cal = y_data.reshape(-1)

            elif y_data.ndim == 2:

                if y_data.shape[1] != y_dim:
                    raise ValueError(
                        f"Dataset {data_id} has "
                        f"{y_data.shape[1]} response dimensions, "
                        f"but expected {y_dim}."
                    )

                yj_cal = y_data[:, j]

            else:
                raise ValueError(
                    f"Response array for dataset {data_id} must be "
                    "one-dimensional or two-dimensional."
                )

            yj_cal_all.append(
                yj_cal
            )

        # ----------------------------------------------
        # Run radius bisection tests
        # ----------------------------------------------

        (
            loss_results_j_all,
            p_values_j_all,
            rejections_j_all,
            best_rejections_j,
        ) = run_radius_tests_for_models(
            predictor_models=predictor_j_models,
            X_cal_all=X_cal_all,
            y_cal_all=yj_cal_all,
            a_values_all=a_values_dim_all[j],
            eps=eps,
            alpha=alpha,
            tol=tol,
            max_iter=max_iter,
            verbose=verbose,
            label=f"Y{j + 1}",
        )

        loss_results_dim_all.append(
            loss_results_j_all
        )

        p_values_dim_all.append(
            p_values_j_all
        )

        rejections_dim_all.append(
            rejections_j_all
        )

        best_rejections_dim.append(
            best_rejections_j
        )

    return (
        loss_results_dim_all,
        p_values_dim_all,
        rejections_dim_all,
        best_rejections_dim,
    )

In [10]:
#quantiles training
def train_quantile_models_for_predictors(
    predictor_models,
    X_train_all,
    y_train_all,
    architecture,
    tau_lower,
    tau_upper,
    epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    verbose=True,
    label="",
):
    quantile_lower_models = []
    quantile_upper_models = []

    lower_histories = []
    upper_histories = []

    for data_id in range(len(predictor_models)):

        if verbose:
            print("\n==============================")
            print(f"Quantile training {label}: Dataset {data_id}")
            print("==============================")

        predictor = predictor_models[data_id]
        X_train = X_train_all[data_id]
        y_train = y_train_all[data_id]

        if y_train.ndim == 1:
            y_train = y_train.reshape(-1, 1)

        input_dim = X_train.shape[1]

        lower_quantile_model = build_lower_quantile_model(
            predictor=predictor,
            input_dim=input_dim,
            architecture=architecture,
        )

        upper_quantile_model = build_upper_quantile_model(
            predictor=predictor,
            input_dim=input_dim,
            architecture=architecture,
        )

        lower_quantile_model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss=pinball_loss(tau_lower),
        )

        upper_quantile_model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss=pinball_loss(tau_upper),
        )

        lower_history = lower_quantile_model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
        )

        upper_history = upper_quantile_model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
        )

        quantile_lower_models.append(lower_quantile_model)
        quantile_upper_models.append(upper_quantile_model)

        lower_histories.append(lower_history)
        upper_histories.append(upper_history)

    return (
        quantile_lower_models,
        quantile_upper_models,
        lower_histories,
        upper_histories,
    )


def train_all_quantile_models(
    predictor_dim_models,
    X_train_all,
    y_train_all,
    architecture,
    tau_lower,
    tau_upper,
    epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    verbose=True,
):
    num_datasets = len(X_train_all)

    Y0 = y_train_all[0]

    if Y0.ndim == 1:
        y_dim = 1
    else:
        y_dim = Y0.shape[1]

    quantile_lower_dim_models = []
    quantile_upper_dim_models = []

    lower_histories_dim = []
    upper_histories_dim = []

    # --------------------------------------------------
    # Separate predictors: Y_j quantiles
    # --------------------------------------------------

    for j in range(y_dim):

        predictor_j_models = [
            predictor_dim_models[data_id][j]
            for data_id in range(num_datasets)
        ]

        yj_train_all = [
            y_train_all[data_id].reshape(-1, 1)
            if y_train_all[data_id].ndim == 1
            else y_train_all[data_id][:, j:j + 1]
            for data_id in range(num_datasets)
        ]

        (
            quantile_lower_j_models,
            quantile_upper_j_models,
            lower_histories_j,
            upper_histories_j,
        ) = train_quantile_models_for_predictors(
            predictor_models=predictor_j_models,
            X_train_all=X_train_all,
            y_train_all=yj_train_all,
            architecture=architecture,
            tau_lower=tau_lower,
            tau_upper=tau_upper,
            epochs=epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
            verbose=verbose,
            label=f"Y{j + 1}",
        )

        quantile_lower_dim_models.append(
            quantile_lower_j_models
        )
        quantile_upper_dim_models.append(
            quantile_upper_j_models
        )

        lower_histories_dim.append(
            lower_histories_j
        )
        upper_histories_dim.append(
            upper_histories_j
        )

    return (
        quantile_lower_dim_models,
        quantile_upper_dim_models,
        lower_histories_dim,
        upper_histories_dim,
    )

In [11]:
def quantile_losses_over_scales(
    predictor,
    lower_quantile_model,
    upper_quantile_model,
    X,
    Y,
    i_values,
):
    Y = np.asarray(Y).reshape(-1)

    f_hat = np.asarray(
        predictor.predict(X, verbose=0)
    ).reshape(-1)

    lower_base = np.asarray(
        lower_quantile_model.predict(X, verbose=0)
    ).reshape(-1)

    upper_base = np.asarray(
        upper_quantile_model.predict(X, verbose=0)
    ).reshape(-1)

    q_l = f_hat - lower_base
    q_u = upper_base - f_hat

    losses = {}

    for i in i_values:
        lower_i = f_hat - i * q_l
        upper_i = f_hat + i * q_u

        losses[i] = np.mean(
            (Y < lower_i) | (Y > upper_i)
        )

    return losses


def run_quantile_hypothesis_test(
    predictor,
    lower_quantile_model,
    upper_quantile_model,
    X_cal,
    y_cal,
    i_values,
    eps,
    alpha,
    tol=1e-4,
    max_iter=100,
    verbose=False,
):
    """
    Run quantile prediction-set hypothesis testing
    using bisection over i.
    """

    X_cal = np.asarray(X_cal)
    y_cal = np.asarray(y_cal).reshape(-1)

    # --------------------------------------------------
    # Compute model outputs once
    # --------------------------------------------------

    f_hat = np.asarray(
        predictor.predict(
            X_cal,
            verbose=0,
        )
    ).reshape(-1)

    lower_base = np.asarray(
        lower_quantile_model.predict(
            X_cal,
            verbose=0,
        )
    ).reshape(-1)

    upper_base = np.asarray(
        upper_quantile_model.predict(
            X_cal,
            verbose=0,
        )
    ).reshape(-1)

    # --------------------------------------------------
    # Distances from predictor to quantile models
    # --------------------------------------------------

    q_l = f_hat - lower_base
    q_u = upper_base - f_hat

    # --------------------------------------------------
    # Prediction-set bounds
    # --------------------------------------------------

    def lower_bound_fn(X, i):
        return f_hat - i * q_l

    def upper_bound_fn(X, i):
        return f_hat + i * q_u

    # --------------------------------------------------
    # Initial bisection interval
    # --------------------------------------------------

    i_min = float(np.min(i_values))
    i_max = float(np.max(i_values))

    # --------------------------------------------------
    # Run bisection
    # --------------------------------------------------

    (
        losses,
        p_values,
        rejected_i,
        best_rejection,
    ) = bisection_prediction_set(
        X=X_cal,
        Y=y_cal,
        lower_bound_fn=lower_bound_fn,
        upper_bound_fn=upper_bound_fn,
        i_min=i_min,
        i_max=i_max,
        eps=eps,
        alpha=alpha,
        tol=tol,
        max_iter=max_iter,
        verbose=verbose,
    )

    return (
        losses,
        p_values,
        rejected_i,
        best_rejection,
    )


def run_quantile_tests_for_models(
    predictor_models,
    lower_quantile_models,
    upper_quantile_models,
    X_cal_all,
    y_cal_all,
    i_values_all,
    eps,
    alpha,
    tol=1e-4,
    max_iter=100,
    verbose=True,
    label="",
):
    loss_results_all = []
    p_values_all = []
    rejections_all = []
    best_rejections = []

    for data_id in range(len(predictor_models)):

        if verbose:
            print("\n==============================")
            print(
                f"Quantile calibration testing {label}: "
                f"Dataset {data_id}"
            )
            print("==============================")

        (
            losses,
            p_values,
            rejected_i,
            best_rejection,
        ) = run_quantile_hypothesis_test(
            predictor=predictor_models[data_id],
            lower_quantile_model=lower_quantile_models[data_id],
            upper_quantile_model=upper_quantile_models[data_id],
            X_cal=X_cal_all[data_id],
            y_cal=y_cal_all[data_id],
            i_values=i_values_all[data_id],
            eps=eps,
            alpha=alpha,
            tol=tol,
            max_iter=max_iter,
            verbose=verbose,
        )

        loss_results_all.append(losses)
        p_values_all.append(p_values)
        rejections_all.append(rejected_i)
        best_rejections.append(best_rejection)

    return (
        loss_results_all,
        p_values_all,
        rejections_all,
        best_rejections,
    )

def run_all_quantile_tests(
    predictor_dim_models,
    quantile_lower_dim_models,
    quantile_upper_dim_models,
    X_cal_all,
    y_cal_all,
    i_values_all,
    eps,
    alpha,
    tol=1e-4,
    max_iter=100,
    verbose=True,
):
    num_datasets = len(X_cal_all)

    Y0 = np.asarray(y_cal_all[0])

    if Y0.ndim == 1:
        y_dim = 1
    else:
        y_dim = Y0.shape[1]

    loss_results_dim_all = []
    p_values_dim_all = []
    rejections_dim_all = []
    best_rejections_dim = []

    for j in range(y_dim):

        predictor_j_models = [
            predictor_dim_models[data_id][j]
            for data_id in range(num_datasets)
        ]

        yj_cal_all = [
            np.asarray(y_cal_all[data_id]).reshape(-1)
            if np.asarray(y_cal_all[data_id]).ndim == 1
            else np.asarray(y_cal_all[data_id])[:, j]
            for data_id in range(num_datasets)
        ]

        (
            loss_results_j_all,
            p_values_j_all,
            rejections_j_all,
            best_rejections_j,
        ) = run_quantile_tests_for_models(
            predictor_models=predictor_j_models,
            lower_quantile_models=quantile_lower_dim_models[j],
            upper_quantile_models=quantile_upper_dim_models[j],
            X_cal_all=X_cal_all,
            y_cal_all=yj_cal_all,
            i_values_all=i_values_all,
            eps=eps,
            alpha=alpha,
            tol=tol,
            max_iter=max_iter,
            verbose=verbose,
            label=f"Y{j + 1}",
        )

        loss_results_dim_all.append(
            loss_results_j_all
        )

        p_values_dim_all.append(
            p_values_j_all
        )

        rejections_dim_all.append(
            rejections_j_all
        )

        best_rejections_dim.append(
            best_rejections_j
        )

    return (
        loss_results_dim_all,
        p_values_dim_all,
        rejections_dim_all,
        best_rejections_dim,
    )

In [12]:
def indicator_losses_over_quantile_scales(
    lower_quantile_models,
    upper_quantile_models,
    X,
    Y,
    i_values,
):
    """
    Compute indicator losses over scale values i for a
    one-dimensional response.
    """

    Y = np.asarray(Y).reshape(-1)

    lower_pred = (
        lower_quantile_models.predict(X, verbose=0)
        .reshape(-1)
    )

    upper_pred = (
        upper_quantile_models.predict(X, verbose=0)
        .reshape(-1)
    )

    losses = {}

    for i in i_values:

        lower_i = i * lower_pred
        upper_i = i * upper_pred

        losses[i] = np.mean(
            (Y < lower_i) | (Y > upper_i)
        )

    return losses

In [13]:
# ==================================================
# Architectures
# ==================================================

predictor_architectures = [
    [(128, "relu"),(128, tf.keras.activations.gelu), (128, tf.keras.activations.gelu)],  # Y1
    [(128, "relu"), (128, tf.keras.activations.gelu), (128, tf.keras.activations.gelu)],    # Y2
]


real_data_architectures = [
    [
        (64, 'elu'), 
        (32, 'elu'), 
        (32, tf.keras.activations.gelu), 
        (32, 'elu'),
        (32, 'relu')
    ],
    [
        (64, "elu"),
        (32, "elu"),
        (32, "elu"),
        (32, "elu"),
        (32, "elu")
    ],
    [
        (64, "relu"),
        (32, tf.keras.activations.gelu),
        (32, tf.keras.activations.gelu),
        (128, 'relu')
    ],

    [
        (32, "elu"), 
        (32, "relu"),
        (32, "relu"),
        (64, tf.keras.activations.gelu)
    ]
]

y1_generator_architecture =[(16, tf.keras.activations.gelu),
                            (16, 'elu'),
                            (16, 'elu'),
                            (16, 'elu'),
                            (512, tf.keras.activations.gelu),
                            (64, "relu")
                           ]

y2_generator_architecture = [(32, tf.keras.activations.gelu),
                             (16, 'relu'),
                             (32, tf.keras.activations.gelu),
                             (32, 'elu'),
                             (512, tf.keras.activations.gelu)
                            ] 


# ==================================================
# Main parameters
# ==================================================
N=1000
n = 20000
c = 2000

data_dimension = 2
noise_dim = 100

alpha = 0.1
eps = 0.1

max_bisection_iter = 100


# ==================================================
# Generate training and calibration datasets
# ==================================================

(
    real_generators,
    generated_datasets,
    full_generated_datasets,
    X_train_all,
    X_cal_all,
    y1_generator,
    y2_generator,
    y_train_all,
    y_cal_all,
) = generate_architecture_datasets(
    real_data_architectures=real_data_architectures,
    data_dimension=data_dimension,
    noise_dim=noise_dim,
    n=n,
    c=c,
    y1_generator_architecture=y1_generator_architecture,
    y2_generator_architecture=y2_generator_architecture,
    seed_base=2026,
    verbose=True,
)


# ==================================================
# Check dataset alignment
# ==================================================

num_datasets = len(generated_datasets)

if len(full_generated_datasets) != num_datasets:
    raise ValueError(
        "full_generated_datasets must contain exactly one response "
        "dataset for each generated feature dataset."
    )

if len(X_train_all) != num_datasets:
    raise ValueError(
        "X_train_all must contain exactly one training feature array "
        "for each generated dataset."
    )

if len(X_cal_all) != num_datasets:
    raise ValueError(
        "X_cal_all must contain exactly one calibration feature array "
        "for each generated dataset."
    )

if len(y_train_all) != num_datasets:
    raise ValueError(
        "y_train_all must contain exactly one training response array "
        "for each generated dataset."
    )

if len(y_cal_all) != num_datasets:
    raise ValueError(
        "y_cal_all must contain exactly one calibration response array "
        "for each generated dataset."
    )


# ==================================================
# Train separate predictors for Y1 and Y2
# ==================================================

(
    predictor_dim_models,
    history_dim,
) = train_all_predictors(
    X_train_all=X_train_all,
    y_train_all=y_train_all,
    predictor_architectures=predictor_architectures,
    epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    validation_split=0.2,
    verbose=0,
)


# ==================================================
# Construct radius search grids
#
# IMPORTANT:
# These are now used only to define the initial
# bisection ranges and desired numerical resolution.
#
# We DO NOT calculate the calibration loss for
# every value in these grids.
# ==================================================

(
    a_min_dim_all,
    a_max_dim_all,
    a_values_dim_all,
) = compute_all_radius_grids(
    predictor_dim_models=predictor_dim_models,
    X_train_all=X_train_all,
    y_train_all=y_train_all,
    N=N,
    lower_scale=0.8,
    upper_scale=1.2,
    verbose=True,
)


# ==================================================
# Radius bisection tolerance
#

# ==================================================

radius_tol = min(
    np.min(np.diff(np.asarray(a_values)))
    for a_values_j in a_values_dim_all
    for a_values in a_values_j
)

print(
    f"\nRadius bisection tolerance: {radius_tol:.8f}"
)


# ==================================================
# Run radius calibration hypothesis tests
# using BISECTION
# ==================================================

(
    loss_results_dim_radius,
    p_values_dim_radius,
    rejections_dim_radius,
    best_rejections_dim_radius,
) = run_all_radius_tests(
    predictor_dim_models=predictor_dim_models,
    X_cal_all=X_cal_all,
    y_cal_all=y_cal_all,
    a_values_dim_all=a_values_dim_all,
    eps=eps,
    alpha=alpha,
    tol=radius_tol,
    max_iter=max_bisection_iter,
    verbose=True,
)


# ==================================================
# Train lower and upper quantile models
# ==================================================

tau_lower = eps/2
tau_upper =1-tau_lower

quantile_architecture = [
    (32, "relu"),
    (32, tf.keras.activations.gelu),
    (32, tf.keras.activations.gelu),
    (32, tf.keras.activations.gelu),
    (64, "relu"),
]

(
    quantile_lower_dim_models,
    quantile_upper_dim_models,
    lower_histories_dim,
    upper_histories_dim,
) = train_all_quantile_models(
    predictor_dim_models=predictor_dim_models,
    X_train_all=X_train_all,
    y_train_all=y_train_all,
    architecture=quantile_architecture,
    tau_lower=tau_lower,
    tau_upper=tau_upper,
    epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    verbose=True,
)


# ==================================================
# Construct quantile-scale search grids
#
# ==================================================

i_values_all = [
    np.linspace(0.5, 1.5, N)
    for _ in range(num_datasets)
]


# ==================================================
# Quantile-scale bisection tolerance
# ==================================================

quantile_tol = min(
    np.min(np.diff(np.asarray(i_values)))
    for i_values in i_values_all
)

print(
    f"Quantile bisection tolerance: {quantile_tol:.8f}"
)


# ==================================================
# Run quantile-band calibration hypothesis tests
# using BISECTION
# ==================================================

(
    loss_results_dim_quantile,
    p_values_dim_quantile,
    rejections_dim_quantile,
    best_rejections_dim_quantile,
) = run_all_quantile_tests(
    predictor_dim_models=predictor_dim_models,
    quantile_lower_dim_models=quantile_lower_dim_models,
    quantile_upper_dim_models=quantile_upper_dim_models,
    X_cal_all=X_cal_all,
    y_cal_all=y_cal_all,
    i_values_all=i_values_all,
    eps=eps,
    alpha=alpha,
    tol=quantile_tol,
    max_iter=max_bisection_iter,
    verbose=True,
)


Dataset 0
Seed = 2026

Feature data shape = (22000, 2)

Dataset 1
Seed = 2027
Feature data shape = (22000, 2)

Dataset 2
Seed = 2028
Feature data shape = (22000, 2)

Dataset 3
Seed = 2029
Feature data shape = (22000, 2)

Feature Dataset 0
Full dataset 0: [X, Y1, Y2]
Shape = (22000, 4)
[[-6.845608   -0.72990173  0.19130297 -0.2013838 ]
 [-5.39727    -7.6722236   0.1070595  -0.3884074 ]
 [-8.280154   -4.9700713   0.13626856 -0.42696357]
 [ 2.0270417  -7.806521    0.22859824 -0.30684787]
 [-8.546704   -9.268836    0.14078712 -0.47822255]]

Feature Dataset 1
Full dataset 1: [X, Y1, Y2]
Shape = (22000, 4)
[[-4.0691595e+00 -9.2862549e+00  9.5806360e-02 -4.6308827e-01]
 [ 9.7621174e+00  9.6762705e+00  1.6394883e-02 -1.8584068e-01]
 [ 9.2560081e+00  9.7611675e+00  1.2041868e-02 -1.8693137e-01]
 [ 8.7388210e+00  4.3611670e+00 -4.3925345e-03 -1.7933220e-01]
 [ 9.5727787e+00 -5.5427942e+00  1.8963908e-01 -3.8465452e-01]]

Feature Dataset 2
Full dataset 2: [X, Y1, Y2]
Shape = (22000, 4)
[[ 4.7114

In [16]:
# --------------------------------------------------
# Output folder
# --------------------------------------------------

output_dir = "combined_prediction_results"
os.makedirs(output_dir, exist_ok=True)


# --------------------------------------------------
# Helper: format selected radius
# --------------------------------------------------

def format_radius(a_best):
    if a_best is None:
        return "not certified"

    return f"{a_best:.4f}"


# --------------------------------------------------
# Helper: format selected quantile scale
# --------------------------------------------------

def format_scale(i_best):
    if i_best is None:
        return "not certified"

    return f"{i_best:.4f}"


# --------------------------------------------------
# Helper: create point-prediction surface
# --------------------------------------------------

def make_prediction_surface(
    model,
    data,
    grid_size=60,
):
    """
    Evaluate a scalar predictor over an X1-X2 grid.

    Any features beyond X1 and X2 are fixed at their
    sample means.
    """
    data = np.asarray(data)

    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(
            "data must be a two-dimensional array with at least "
            "two feature columns."
        )

    x1 = data[:, 0]
    x2 = data[:, 1]

    x1_grid = np.linspace(
        x1.min(),
        x1.max(),
        grid_size,
    )

    x2_grid = np.linspace(
        x2.min(),
        x2.max(),
        grid_size,
    )

    X1_grid, X2_grid = np.meshgrid(
        x1_grid,
        x2_grid,
    )

    # Fix additional features at their sample means.
    feature_means = np.mean(data, axis=0)

    X_grid = np.tile(
        feature_means,
        (grid_size * grid_size, 1),
    )

    X_grid[:, 0] = X1_grid.ravel()
    X_grid[:, 1] = X2_grid.ravel()

    y_pred_grid = np.asarray(
        model.predict(
            X_grid,
            verbose=0,
        )
    ).reshape(-1)

    expected_size = grid_size * grid_size

    if y_pred_grid.size != expected_size:
        raise ValueError(
            "The prediction model must produce one scalar prediction "
            "for each grid point."
        )

    Y_pred_grid = y_pred_grid.reshape(
        X1_grid.shape
    )

    return (
        X1_grid,
        X2_grid,
        Y_pred_grid,
    )


# --------------------------------------------------
# Helper: create prediction and quantile-band surfaces
# --------------------------------------------------

def make_quantile_band_surface(
    predictor_model,
    lower_quantile_model,
    upper_quantile_model,
    data,
    i_best,
    grid_size=60,
):
    """
    Create the point-prediction, lower-band, and upper-band
    surfaces over a grid of X1 and X2 values.
    """
    if i_best is None:
        raise ValueError(
            "i_best is None because no quantile scale was certified."
        )

    data = np.asarray(data)

    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(
            "data must be a two-dimensional array containing "
            "at least X1 and X2."
        )

    x1 = data[:, 0]
    x2 = data[:, 1]

    x1_grid = np.linspace(
        x1.min(),
        x1.max(),
        grid_size,
    )

    x2_grid = np.linspace(
        x2.min(),
        x2.max(),
        grid_size,
    )

    X1_grid, X2_grid = np.meshgrid(
        x1_grid,
        x2_grid,
    )

    # Fix additional features at their sample means.
    feature_means = np.mean(data, axis=0)

    X_grid = np.tile(
        feature_means,
        (grid_size * grid_size, 1),
    )

    X_grid[:, 0] = X1_grid.ravel()
    X_grid[:, 1] = X2_grid.ravel()

    f_hat = np.asarray(
        predictor_model.predict(
            X_grid,
            verbose=0,
        )
    ).reshape(-1)

    lower_base = np.asarray(
        lower_quantile_model.predict(
            X_grid,
            verbose=0,
        )
    ).reshape(-1)

    upper_base = np.asarray(
        upper_quantile_model.predict(
            X_grid,
            verbose=0,
        )
    ).reshape(-1)

    expected_size = grid_size * grid_size

    if (
        f_hat.size != expected_size
        or lower_base.size != expected_size
        or upper_base.size != expected_size
    ):
        raise ValueError(
            "Each model must produce one scalar prediction "
            "for every grid point."
        )

    # Distances from the point predictor to the base quantiles.
    q_l = f_hat - lower_base
    q_u = upper_base - f_hat

    f_grid = f_hat.reshape(
        X1_grid.shape
    )

    lower_grid = (
        f_hat - i_best * q_l
    ).reshape(
        X1_grid.shape
    )

    upper_grid = (
        f_hat + i_best * q_u
    ).reshape(
        X1_grid.shape
    )

    return (
        X1_grid,
        X2_grid,
        f_grid,
        lower_grid,
        upper_grid,
    )


# --------------------------------------------------
# Check overall structure
# --------------------------------------------------

num_feature_datasets = len(generated_datasets)

if len(full_generated_datasets) != num_feature_datasets:
    raise ValueError(
        "full_generated_datasets must contain exactly one response "
        "dataset for each feature dataset."
    )

if len(predictor_dim_models) != num_feature_datasets:
    raise ValueError(
        "predictor_dim_models must contain exactly one model "
        "collection for each feature dataset."
    )

if len(best_rejections_dim_radius) < 2:
    raise ValueError(
        "best_rejections_dim_radius must contain results for "
        "Y1 and Y2."
    )

if len(best_rejections_dim_quantile) < 2:
    raise ValueError(
        "best_rejections_dim_quantile must contain results for "
        "Y1 and Y2."
    )

if len(quantile_lower_dim_models) < 2:
    raise ValueError(
        "quantile_lower_dim_models must contain models for "
        "Y1 and Y2."
    )

if len(quantile_upper_dim_models) < 2:
    raise ValueError(
        "quantile_upper_dim_models must contain models for "
        "Y1 and Y2."
    )


# --------------------------------------------------
# Create one combined HTML figure per feature dataset
# --------------------------------------------------

for feature_id in range(num_feature_datasets):

    dataset_id = feature_id

    dataset_with_y = np.asarray(
        full_generated_datasets[dataset_id]
    )

    if dataset_with_y.ndim != 2:
        raise ValueError(
            f"full_generated_datasets[{dataset_id}] must be "
            "a two-dimensional array."
        )

    if dataset_with_y.shape[1] < 4:
        raise ValueError(
            f"full_generated_datasets[{dataset_id}] must contain "
            "columns X1, X2, Y1, and Y2."
        )

    if len(predictor_dim_models[dataset_id]) < 2:
        raise ValueError(
            f"predictor_dim_models[{dataset_id}] must contain "
            "predictor models for Y1 and Y2."
        )

    for response_dim in range(2):

        if len(best_rejections_dim_radius[response_dim]) <= dataset_id:
            raise ValueError(
                f"Missing radius result for Y{response_dim + 1} "
                f"of dataset {dataset_id}."
            )

        if len(best_rejections_dim_quantile[response_dim]) <= dataset_id:
            raise ValueError(
                f"Missing quantile result for Y{response_dim + 1} "
                f"of dataset {dataset_id}."
            )

        if len(quantile_lower_dim_models[response_dim]) <= dataset_id:
            raise ValueError(
                f"Missing lower quantile model for Y{response_dim + 1} "
                f"of dataset {dataset_id}."
            )

        if len(quantile_upper_dim_models[response_dim]) <= dataset_id:
            raise ValueError(
                f"Missing upper quantile model for Y{response_dim + 1} "
                f"of dataset {dataset_id}."
            )

    # --------------------------------------------------
    # Extract feature and response data
    # --------------------------------------------------

    X_plot = dataset_with_y[:, :2]

    y1 = dataset_with_y[:, 2]
    y2 = dataset_with_y[:, 3]

    # Point-prediction models
    y1_model = predictor_dim_models[dataset_id][0]
    y2_model = predictor_dim_models[dataset_id][1]

    # Certified radii
    a_y1 = best_rejections_dim_radius[0][dataset_id]
    a_y2 = best_rejections_dim_radius[1][dataset_id]

    # Quantile models
    y1_lower_model = quantile_lower_dim_models[0][dataset_id]
    y1_upper_model = quantile_upper_dim_models[0][dataset_id]

    y2_lower_model = quantile_lower_dim_models[1][dataset_id]
    y2_upper_model = quantile_upper_dim_models[1][dataset_id]

    # Certified quantile scales
    i_y1 = best_rejections_dim_quantile[0][dataset_id]
    i_y2 = best_rejections_dim_quantile[1][dataset_id]

    # --------------------------------------------------
    # Create 2-by-3 figure
    #
    # Row 1: Y1
    # Row 2: Y2
    #
    # Column 1: observations only
    # Column 2: radius-based prediction band
    # Column 3: quantile-based prediction band
    # --------------------------------------------------

    fig = make_subplots(
        rows=2,
        cols=3,
        specs=[
            [
                {"type": "scene"},
                {"type": "scene"},
                {"type": "scene"},
            ],
            [
                {"type": "scene"},
                {"type": "scene"},
                {"type": "scene"},
            ],
        ],
        subplot_titles=(
            "Y1 observations",
            f"Y1 radius band, a={format_radius(a_y1)}",
            f"Y1 quantile band, i={format_scale(i_y1)}",
            "Y2 observations",
            f"Y2 radius band, a={format_radius(a_y2)}",
            f"Y2 quantile band, i={format_scale(i_y2)}",
        ),
        horizontal_spacing=0.001,
        vertical_spacing=0.015,
    )

    for annotation in fig.layout.annotations:
        annotation.font.size = 24

    response_information = [
        {
            "response": y1,
            "response_name": "Y1",
            "model": y1_model,
            "a_best": a_y1,
            "lower_model": y1_lower_model,
            "upper_model": y1_upper_model,
            "i_best": i_y1,
            "row": 1,
        },
        {
            "response": y2,
            "response_name": "Y2",
            "model": y2_model,
            "a_best": a_y2,
            "lower_model": y2_lower_model,
            "upper_model": y2_upper_model,
            "i_best": i_y2,
            "row": 2,
        },
    ]

    for response_info in response_information:

        response = response_info["response"]
        response_name = response_info["response_name"]
        model = response_info["model"]
        a_best = response_info["a_best"]
        lower_model = response_info["lower_model"]
        upper_model = response_info["upper_model"]
        i_best = response_info["i_best"]
        row = response_info["row"]

        # --------------------------------------------------
        # Column 1: observations only
        # --------------------------------------------------

        fig.add_trace(
            go.Scatter3d(
                x=dataset_with_y[:, 0],
                y=dataset_with_y[:, 1],
                z=response,
                mode="markers",
                marker=dict(
                    size=2,
                    opacity=0.35,
                ),
                name=f"{response_name} observations",
                legendgroup=response_name,
                showlegend=False,
            ),
            row=row,
            col=1,
        )

        # --------------------------------------------------
        # Column 2: observations with radius-based band
        # --------------------------------------------------

        fig.add_trace(
            go.Scatter3d(
                x=dataset_with_y[:, 0],
                y=dataset_with_y[:, 1],
                z=response,
                mode="markers",
                marker=dict(
                    size=2,
                    opacity=0.25,
                ),
                name=f"{response_name} radius observations",
                legendgroup=f"{response_name}_radius",
                showlegend=False,
            ),
            row=row,
            col=2,
        )

        (
            X1_radius_grid,
            X2_radius_grid,
            prediction_radius_grid,
        ) = make_prediction_surface(
            model=model,
            data=X_plot,
            grid_size=60,
        )

        # Point predictor is shown even if no radius was certified.
        fig.add_trace(
            go.Surface(
                x=X1_radius_grid,
                y=X2_radius_grid,
                z=prediction_radius_grid,
                opacity=0.65,
                showscale=False,
                name=f"{response_name} predictor",
                legendgroup=f"{response_name}_radius",
                showlegend=False,
            ),
            row=row,
            col=2,
        )

        if a_best is not None:

            fig.add_trace(
                go.Surface(
                    x=X1_radius_grid,
                    y=X2_radius_grid,
                    z=prediction_radius_grid - a_best,
                    opacity=0.25,
                    showscale=False,
                    name=f"{response_name} radius lower",
                    legendgroup=f"{response_name}_radius",
                    showlegend=False,
                ),
                row=row,
                col=2,
            )

            fig.add_trace(
                go.Surface(
                    x=X1_radius_grid,
                    y=X2_radius_grid,
                    z=prediction_radius_grid + a_best,
                    opacity=0.25,
                    showscale=False,
                    name=f"{response_name} radius upper",
                    legendgroup=f"{response_name}_radius",
                    showlegend=False,
                ),
                row=row,
                col=2,
            )

        # --------------------------------------------------
        # Column 3: observations with quantile-based band
        # --------------------------------------------------

        fig.add_trace(
            go.Scatter3d(
                x=dataset_with_y[:, 0],
                y=dataset_with_y[:, 1],
                z=response,
                mode="markers",
                marker=dict(
                    size=2,
                    opacity=0.25,
                ),
                name=f"{response_name} quantile observations",
                legendgroup=f"{response_name}_quantile",
                showlegend=False,
            ),
            row=row,
            col=3,
        )

        if i_best is not None:

            (
                X1_quantile_grid,
                X2_quantile_grid,
                prediction_quantile_grid,
                lower_quantile_grid,
                upper_quantile_grid,
            ) = make_quantile_band_surface(
                predictor_model=model,
                lower_quantile_model=lower_model,
                upper_quantile_model=upper_model,
                data=X_plot,
                i_best=i_best,
                grid_size=60,
            )

            fig.add_trace(
                go.Surface(
                    x=X1_quantile_grid,
                    y=X2_quantile_grid,
                    z=prediction_quantile_grid,
                    opacity=0.65,
                    showscale=False,
                    name=f"{response_name} quantile predictor",
                    legendgroup=f"{response_name}_quantile",
                    showlegend=False,
                ),
                row=row,
                col=3,
            )

            fig.add_trace(
                go.Surface(
                    x=X1_quantile_grid,
                    y=X2_quantile_grid,
                    z=lower_quantile_grid,
                    opacity=0.25,
                    showscale=False,
                    name=f"{response_name} quantile lower",
                    legendgroup=f"{response_name}_quantile",
                    showlegend=False,
                ),
                row=row,
                col=3,
            )

            fig.add_trace(
                go.Surface(
                    x=X1_quantile_grid,
                    y=X2_quantile_grid,
                    z=upper_quantile_grid,
                    opacity=0.25,
                    showscale=False,
                    name=f"{response_name} quantile upper",
                    legendgroup=f"{response_name}_quantile",
                    showlegend=False,
                ),
                row=row,
                col=3,
            )

        else:
            # Show the point predictor even when no quantile scale
            # was certified.
            (
                X1_quantile_grid,
                X2_quantile_grid,
                prediction_quantile_grid,
            ) = make_prediction_surface(
                model=model,
                data=X_plot,
                grid_size=60,
            )

            fig.add_trace(
                go.Surface(
                    x=X1_quantile_grid,
                    y=X2_quantile_grid,
                    z=prediction_quantile_grid,
                    opacity=0.65,
                    showscale=False,
                    name=f"{response_name} quantile predictor",
                    legendgroup=f"{response_name}_quantile",
                    showlegend=False,
                ),
                row=row,
                col=3,
            )

    # --------------------------------------------------
    # Use the same camera angle for all six panels
    # --------------------------------------------------

    camera = dict(
        eye=dict(
            x=1.5,
            y=1.5,
            z=1.1,
        )
    )

    scene_names = [
        "scene",
        "scene2",
        "scene3",
        "scene4",
        "scene5",
        "scene6",
    ]

    z_titles = [
        "Y1",
        "Y1",
        "Y1",
        "Y2",
        "Y2",
        "Y2",
    ]

    for scene_name, z_title in zip(
        scene_names,
        z_titles,
    ):
        fig.layout[scene_name].xaxis.title = "X1"
        fig.layout[scene_name].yaxis.title = "X2"
        fig.layout[scene_name].zaxis.title = z_title
        fig.layout[scene_name].camera = camera


    # --------------------------------------------------
    # Make the six 3D subplot domains larger
    # --------------------------------------------------

    scene_domains = {
        "scene":  dict(x=[0.00, 0.33], y=[0.51, 0.98]),
        "scene2": dict(x=[0.335, 0.665], y=[0.51, 0.98]),
        "scene3": dict(x=[0.67, 1.00], y=[0.51, 0.98]),

        "scene4": dict(x=[0.00, 0.33], y=[0.00, 0.47]),
        "scene5": dict(x=[0.335, 0.665], y=[0.00, 0.47]),
        "scene6": dict(x=[0.67, 1.00], y=[0.00, 0.47]),
    }

    for scene_name, domain in scene_domains.items():
        fig.layout[scene_name].domain = domain


    # --------------------------------------------------
    # Figure layout
    # --------------------------------------------------

    fig.update_layout(
        title=dict(
            text=(
                f"Feature Dataset {feature_id}: "
                "Observed Responses, Radius Bands, and Quantile Bands"
            ),
            font=dict(size=30),
            x=0.5,
            xanchor="center",
        ),
        width=2400,
        height=1450,
        margin=dict(
            l=5,
            r=5,
            t=80,
            b=5,
        ),
    )
    # --------------------------------------------------
    # Save one combined interactive HTML file
    # --------------------------------------------------

    output_filename = os.path.join(
        output_dir,
        f"feature_dataset_{feature_id}_combined_bands.html",
    )

    fig.write_html(
        output_filename,
        auto_open=False,
    )

    print(f"Saved: {output_filename}")

Saved: combined_prediction_results\feature_dataset_0_combined_bands.html
Saved: combined_prediction_results\feature_dataset_1_combined_bands.html
Saved: combined_prediction_results\feature_dataset_2_combined_bands.html
Saved: combined_prediction_results\feature_dataset_3_combined_bands.html


In [ ]:
# ==================================================
# Evaluate selected radius / quantile values
# on a large independent truth sample
# ==================================================

def evaluate_truth_at_selected_values(
    real_generators,
    y1_generator,
    y2_generator,
    predictor_dim_models,
    quantile_lower_dim_models,
    quantile_upper_dim_models,
    best_rejections_dim_radius,
    best_rejections_dim_quantile,
    m_truth,
    noise_dim,
    eps,
    seed_base=50000,
    batch_size=4096,
    verbose=True,
):

    num_datasets = len(real_generators)
    y_dim = 2

    # One-sided 95% normal critical value
    z_95 = 1.6448536269514722

    results = []

    # ==================================================
    # DATASET LOOP
    # ==================================================

    for data_id, real_gen in enumerate(real_generators):

        seed = seed_base + data_id

        if verbose:
            print("\n")
            print("=" * 70)
            print(f"TRUTH EVALUATION: DATASET {data_id}")
            print(f"m_truth = {m_truth:,}")
            print("=" * 70)

        # ==================================================
        # Generate fresh X sample
        # ==================================================

        feature_noise = tf.random.stateless_normal(
            shape=(m_truth, noise_dim),
            seed=[seed, 0],
        )

        X_truth = real_gen.predict(
            feature_noise,
            batch_size=batch_size,
            verbose=0,
        )

        X_truth = np.asarray(
            X_truth,
            dtype=np.float32,
        )

        del feature_noise

        # ==================================================
        # Generate fresh Y1 and Y2
        #
        # Same data-generation construction as before:
        # y_input = [X1, X2, Z]
        # Y1 and Y2 use the same Z.
        # ==================================================

        rng = np.random.default_rng(seed)

        z_truth = rng.normal(
            size=m_truth
        ).astype(np.float32)

        y_input_truth = np.column_stack([
            X_truth[:, 0],
            X_truth[:, 1],
            z_truth,
        ]).astype(np.float32)

        Y1_truth = np.asarray(
            y1_generator.predict(
                y_input_truth,
                batch_size=batch_size,
                verbose=0,
            )
        ).reshape(-1)

        Y2_truth = np.asarray(
            y2_generator.predict(
                y_input_truth,
                batch_size=batch_size,
                verbose=0,
            )
        ).reshape(-1)

        del z_truth
        del y_input_truth

        Y_truth_list = [
            Y1_truth,
            Y2_truth,
        ]

        # ==================================================
        # RESPONSE LOOP: Y1, Y2
        # ==================================================

        for j in range(y_dim):

            Y_truth = Y_truth_list[j]

            # Predictor indexing from your previous code
            predictor = predictor_dim_models[data_id][j]

            # Evaluate predictor once
            f_hat = np.asarray(
                predictor.predict(
                    X_truth,
                    batch_size=batch_size,
                    verbose=0,
                )
            ).reshape(-1)

            # ==================================================
            # RADIUS METHOD
            # ==================================================

            a_star = best_rejections_dim_radius[j][data_id]

            if a_star is None:
                raise ValueError(
                    f"No selected radius for "
                    f"Dataset {data_id + 1}, Y{j + 1}."
                )

            a_star = float(a_star)

            radius_lower = f_hat - a_star
            radius_upper = f_hat + a_star

            radius_miscoverage = (
                (Y_truth < radius_lower)
                |
                (Y_truth > radius_upper)
            )

            # Bernoulli sample average
            p_hat_radius = float(
                np.mean(radius_miscoverage)
            )

            # Standard error
            se_radius = np.sqrt(
                p_hat_radius
                * (1.0 - p_hat_radius)
                / m_truth
            )

            # One-sided 95% lower confidence bound
            radius_lower_95 = max(
                0.0,
                p_hat_radius - z_95 * se_radius
            )

            radius_significantly_above_eps = (
                radius_lower_95 > eps
            )

            # ==================================================
            # QUANTILE METHOD
            # ==================================================

            i_star = best_rejections_dim_quantile[j][data_id]

            if i_star is None:
                raise ValueError(
                    f"No selected quantile index for "
                    f"Dataset {data_id + 1}, Y{j + 1}."
                )

            i_star = float(i_star)

            # Quantile model indexing from your previous code
            lower_model = (
                quantile_lower_dim_models[j][data_id]
            )

            upper_model = (
                quantile_upper_dim_models[j][data_id]
            )

            lower_base = np.asarray(
                lower_model.predict(
                    X_truth,
                    batch_size=batch_size,
                    verbose=0,
                )
            ).reshape(-1)

            upper_base = np.asarray(
                upper_model.predict(
                    X_truth,
                    batch_size=batch_size,
                    verbose=0,
                )
            ).reshape(-1)

            # Same construction used in your
            # quantile bisection procedure
            q_l = f_hat - lower_base
            q_u = upper_base - f_hat

            quantile_lower = (
                f_hat - i_star * q_l
            )

            quantile_upper = (
                f_hat + i_star * q_u
            )

            quantile_miscoverage = (
                (Y_truth < quantile_lower)
                |
                (Y_truth > quantile_upper)
            )

            # Bernoulli sample average
            p_hat_quantile = float(
                np.mean(quantile_miscoverage)
            )

            # Standard error
            se_quantile = np.sqrt(
                p_hat_quantile
                * (1.0 - p_hat_quantile)
                / m_truth
            )

            # One-sided 95% lower confidence bound
            quantile_lower_95 = max(
                0.0,
                p_hat_quantile
                - z_95 * se_quantile
            )

            quantile_significantly_above_eps = (
                quantile_lower_95 > eps
            )

            # ==================================================
            # Store results
            # ==================================================

            results.append({
                "dataset": data_id + 1,
                "response": f"Y{j + 1}",

                "a_star": a_star,
                "radius_estimate": p_hat_radius,
                "radius_lower_95": radius_lower_95,
                "radius_above_eps":
                    radius_significantly_above_eps,

                "i_star": i_star,
                "quantile_estimate": p_hat_quantile,
                "quantile_lower_95": quantile_lower_95,
                "quantile_above_eps":
                    quantile_significantly_above_eps,
            })

            # ==================================================
            # Print result
            # ==================================================

            if verbose:

                print(
                    f"\nDataset {data_id + 1}, Y{j + 1}"
                )

                print(
                    "\nRadius method"
                    f"\n  a*             = {a_star:.6f}"
                    f"\n  risk estimate  = {p_hat_radius:.6f}"
                    f"\n  95% lower bound= {radius_lower_95:.6f}"
                    f"\n  lower > eps    = "
                    f"{radius_significantly_above_eps}"
                )

                print(
                    "\nQuantile method"
                    f"\n  i*             = {i_star:.6f}"
                    f"\n  risk estimate  = {p_hat_quantile:.6f}"
                    f"\n  95% lower bound= {quantile_lower_95:.6f}"
                    f"\n  lower > eps    = "
                    f"{quantile_significantly_above_eps}"
                )


        import gc
        gc.collect()

    return results


# ==================================================
# Run truth evaluation
# ==================================================

m_truth = 100_000

truth_results = evaluate_truth_at_selected_values(
    real_generators=real_generators,
    y1_generator=y1_generator,
    y2_generator=y2_generator,
    predictor_dim_models=predictor_dim_models,
    quantile_lower_dim_models=quantile_lower_dim_models,
    quantile_upper_dim_models=quantile_upper_dim_models,
    best_rejections_dim_radius=best_rejections_dim_radius,
    best_rejections_dim_quantile=best_rejections_dim_quantile,
    m_truth=m_truth,
    noise_dim=noise_dim,
    eps=eps,
    seed_base=50000,
    batch_size=4096,
    verbose=True,
)

In [ ]:
print(r"""
\begin{table}[H]
\centering
\caption{Selected radius and quantile-scale parameters.}
\label{tab:selected_parameters}
\begin{tabular}{cccc}
\hline
Dataset & Response & $a^\star$ & $i^\star$ \\
\hline
""")

for result in truth_results:

    # truth_results stores datasets starting at 1,
    # so subtract 1 for display
    dataset = result["dataset"] - 1
    response = result["response"]

    a_star = result["a_star"]
    i_star = result["i_star"]

    print(
        f"{dataset} & ${response}$ & "
        f"{a_star:.4f} & "
        f"{i_star:.4f} \\\\"
    )

print(r"""
\hline
\end{tabular}
\end{table}
""")


# ==================================================
# LaTeX Table 2:
# Large-sample evaluation
# Dataset numbering: 0, 1, 2, ...
# ==================================================

print(r"""
\begin{table}[H]
\centering
\caption{Large-sample evaluation of the selected radius and quantile-scale parameters.}
\label{tab:truth_evaluation}
\begin{tabular}{cc|ccc|ccc}
\hline
& &
\multicolumn{3}{c|}{Radius Method} &
\multicolumn{3}{c}{Quantile Method} \\
Dataset & Response &
$\widehat{R}(a^\star)$ &
95\% LB &
LB $>\epsilon$ &
$\widehat{R}(i^\star)$ &
95\% LB &
LB $>\epsilon$ \\
\hline
""")

for result in truth_results:

    # truth_results stores datasets starting at 1,
    # so subtract 1 for display
    dataset = result["dataset"] - 1
    response = result["response"]

    radius_estimate = result["radius_estimate"]
    radius_lower = result["radius_lower_95"]
    radius_above = result["radius_above_eps"]

    quantile_estimate = result["quantile_estimate"]
    quantile_lower = result["quantile_lower_95"]
    quantile_above = result["quantile_above_eps"]

    radius_text = "Yes" if radius_above else "No"
    quantile_text = "Yes" if quantile_above else "No"

    print(
        f"{dataset} & ${response}$ & "
        f"{radius_estimate:.4f} & "
        f"{radius_lower:.4f} & "
        f"{radius_text} & "
        f"{quantile_estimate:.4f} & "
        f"{quantile_lower:.4f} & "
        f"{quantile_text} \\\\"
    )

print(r"""
\hline
\end{tabular}
\end{table}
""")

In [ ]:
headers = [
    "Dataset",
    "Y",
    "a*",
    "R_hat(a*)",
    "95% LB",
    "LB>eps?",
    "i*",
    "R_hat(i*)",
    "95% LB",
    "LB>eps?",
]

# Smaller column widths
widths = [7, 3, 8, 10, 8, 8, 8, 10, 8, 8]

# Header
header_line = " | ".join(
    f"{header:>{width}}"
    for header, width in zip(headers, widths)
)

print(header_line)
print("-" * len(header_line))

# Rows
for result in truth_results:

    # truth_results stores datasets starting at 1,
    # so subtract 1 for display
    dataset = result["dataset"] - 1

    radius_above = (
        "Yes" if result["radius_above_eps"] else "No"
    )

    quantile_above = (
        "Yes" if result["quantile_above_eps"] else "No"
    )

    values = [
        str(dataset),
        result["response"],
        f"{result['a_star']:.4f}",
        f"{result['radius_estimate']:.4f}",
        f"{result['radius_lower_95']:.4f}",
        radius_above,
        f"{result['i_star']:.4f}",
        f"{result['quantile_estimate']:.4f}",
        f"{result['quantile_lower_95']:.4f}",
        quantile_above,
    ]

    print(
        " | ".join(
            f"{value:>{width}}"
            for value, width in zip(values, widths)
        )
    )

In [ ]:
# ==================================================
# LaTeX table:
# Model architectures -- narrower version
# ==================================================

def activation_name(activation):

    if isinstance(activation, str):
        name = activation
    elif hasattr(activation, "__name__"):
        name = activation.__name__
    else:
        name = str(activation)

    name_map = {
        "relu": "ReLU",
        "elu": "ELU",
        "gelu": "GELU",
        "leaky_relu": "Leaky ReLU",
    }

    return name_map.get(name.lower(), name)


def architecture_to_latex(architecture):

    layers = []

    for units, activation in architecture:

        act = activation_name(activation)

        layers.append(
            f"{units} ({act})"
        )

    return r" $\rightarrow$ ".join(layers)


# ==================================================
# Print table
# ==================================================

print(r"""
\begin{table}[H]
\centering
\caption{Neural-network architectures used in the simulation study.}
\label{tab:model_architectures}
\small
\begin{tabular}{p{0.25\textwidth} p{0.65\textwidth}}
\hline
\textbf{Dataset/Response} & \textbf{Hidden-layer architecture} \\
\hline
""")


# Feature generators
for data_id, architecture in enumerate(real_data_architectures):

    print(
        f"Dataset {data_id} & "
        f"{architecture_to_latex(architecture)} \\\\"
    )


# Response generators
print(
    "$Y_1$ (Generator) & "
    f"{architecture_to_latex(y1_generator_architecture)} \\\\"
)

print(
    "$Y_2$ (Generator) & "
    f"{architecture_to_latex(y2_generator_architecture)} \\\\"
)


# Predictors
print(
    "$Y_1$ (Predictor) & "
    f"{architecture_to_latex(predictor_architectures[0])} \\\\"
)

print(
    "$Y_2$ (Predictor) & "
    f"{architecture_to_latex(predictor_architectures[1])} \\\\"
)


# Quantile model
print(
    "Quantile & "
    f"{architecture_to_latex(quantile_architecture)} \\\\"
)


print(r"""
\hline
\end{tabular}
\end{table}
""")